In [3]:
import os
import requests
from bs4 import BeautifulSoup
from datetime import date
import pandas as pd
import json
import shutil
from datetime import datetime
from google.colab import drive

# Montar o Google Drive em um novo diretório
drive.mount('/content/drive2')

# Criar uma pasta chamada "output" se não existir
output_folder = "output"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"\033[92mA pasta {output_folder} foi criada.\033[0m")

# Função para obter os dados dos produtos
def get_product_data(url, headers, page_number):
    try:
        products_data = []
        response = requests.get(f"{url}/page/{page_number}/", headers=headers)

        if response.status_code == 404:
            print(f"\033[91mPágina {page_number} não encontrada (Erro 404).\033[0m")
            return None, False

        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        data_de_coleta = date.today()
        nome_empresa = "Fogões Shop"

        product_list = soup.find_all("li", class_="product-layout")

        if not product_list:
            print(f"Página {page_number} não contém produtos.")
            return None, False

        for product in product_list:
            image_tag = product.find("img", class_="img-responsive")
            image = image_tag["src"] if image_tag else ""
            name_tag = product.find("div", class_="item-title").a
            name = name_tag.text.strip() if name_tag else ""
            price_tag = product.find("span", class_="price")
            price = price_tag.text.strip().replace("R$", "").replace(".", "").replace(",", ".") if price_tag else ""
            currency = "BRL"
            availability = "Disponível"
            product_url = name_tag["href"] if name_tag else ""

            product_data = {
                "Nome empresa": nome_empresa,
                "Data de coleta": data_de_coleta,
                "Imagem": image,
                "Nome": name,
                "Preço": float(price) if price else None,
                "Moeda": currency,
                "Disponibilidade": availability,
                "URL do produto": product_url
            }
            products_data.append(product_data)

        pagination = soup.find("ul", class_="pagination")
        if pagination:
            next_page_link = pagination.find("li", text=">")
            if next_page_link:
                next_page_link = next_page_link.find("a")
            has_next_page = bool(next_page_link)
        else:
            has_next_page = False

        return products_data, has_next_page
    except requests.RequestException as e:
        print(f"\033[91mErro ao obter dados do produto na página {page_number}: {e}\033[0m")
        return None, False

def analyze_xml_links(xml_links, headers):
    for category, link in xml_links.items():
        print("Categoria:", category)
        print("Link:", link)
        product_data = []
        page_number = 1
        while True:
            print(f"Procurando produtos na página {page_number}...")
            data, has_next_page = get_product_data(link, headers, page_number)
            if not data:
                print(f"Parando a busca, nenhuma informação encontrada na página {page_number}.")
                break
            product_data.extend(data)
            if not has_next_page:
                print(f"Última página alcançada: {page_number}.")
                break
            page_number += 1

        if product_data:
            try:
                # Substituir caracteres especiais no nome da categoria
                category = category.replace("/", "-").replace(" ", "_")
                product_df = pd.DataFrame(product_data)
                # Salvar o DataFrame em um arquivo CSV dentro da pasta "output"
                product_df.to_csv(os.path.join(output_folder, f"{category}.csv"), index=False)
                print("Dados do produto salvos em", f"{category}.csv")
            except Exception as e:
                print(f"\033[91mErro ao salvar dados do produto em CSV: {e}\033[0m")
        else:
            print("\033[91mNenhum produto encontrado.\033[0m")
        print()

# Links para as categorias
links = {
    "Acessorios Banheiro": "https://fogoesshop.com.br/acessorios-banheiros-portatoalha-papeleira-prateleira-cabide/acessorios-banheiro/",
    "Modelos de Deca": "https://fogoesshop.com.br/acessorios-banheiros-portatoalha-papeleira-prateleira-cabide/acessorios-modelos-deca/",
    "Modelos Metalworks by Crismoe": "https://fogoesshop.com.br/acessorios-banheiros-portatoalha-papeleira-prateleira-cabide/acessorios-modelos-crismoe-metalworks/",
    "Sacaria": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/sacaria-areia-cimento-argamassa-outros/",
    "Vedacao e impermeabilizacao": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/vedação-impermeabilizante/",
    "Blocos e Tijolos": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/blocos-tijolos/",
    "Papelao, Saco de Entulho e Lona": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/papelao-lona-saco-plastico/",
    "Acos e Madeiras": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/aços-madeiras/",
    "Equipamentos p/ Contrucao": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/equipamento-construcao/",
    "Eletrodutos e Mangueiras": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/eletroduto-conduite-mangueira/",
    "Ferramentas": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/marca-vonder/",
    "Serra-Marmore": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/marca-vonder/Serra-Marmore/",
    "Caixa d'Agua": "https://fogoesshop.com.br/construcao-cimento-areia-ferro-ferramentas/caixa-agua-cisterna/",
    "Tubo e Conexao": "https://fogoesshop.com.br/hidraulica-construcao/tubo-conexao/",
    "Aquecedor gas": "https://fogoesshop.com.br/hidraulica-construcao/Aquecedor-gas/",
    "Bombas e Pressurizadores": "https://fogoesshop.com.br/hidraulica-construcao/bombas-pressurizadores/",
    "Ralo linear": "https://fogoesshop.com.br/hidraulica-construcao/ralo-linear/",
    "Triturador Sanitario": "https://fogoesshop.com.br/hidraulica-construcao/triturador-sanitario-sanitrit/",
    "Caixa de Descarga": "https://fogoesshop.com.br/banheiro-construcao-reforma/fshop-caixa-descarga-embutida-deca/",
    "Colecao Assinada": "https://fogoesshop.com.br/banheiro-construcao-reforma/fshop-colecao-assinada/",
    "Cubas": "https://fogoesshop.com.br/banheiro-construcao-reforma/fshop-cubas/",
    "Louca p/ Banheiros": "https://fogoesshop.com.br/banheiro-construcao-reforma/fshop-loucas-p-banheiros/",
    "Metais p/ Banheiros": "https://fogoesshop.com.br/banheiro-construcao-reforma/fshop-metais-p-banheiros/",
    "Promocoes Banheiro, sale, Black friday": "https://fogoesshop.com.br/banheiro-construcao-reforma/promocoes-banheiro/",
    "Revestir": "https://fogoesshop.com.br/banheiro-construcao-reforma/revestir/",
    "Banheira": "https://fogoesshop.com.br/fshop-banho/banheira/",
    "Duchas e Chuveiros": "https://fogoesshop.com.br/fshop-banho/fshop-duchas/",
    "Chuveiro Eletrico": "https://fogoesshop.com.br/fshop-banho/chuveiro-eletrico/",
    "Mekal": "https://fogoesshop.com.br/fshop-cozinha/cuba-inox-mekal/",
    "Tramontina": "https://fogoesshop.com.br/fshop-cozinha/fshop-tramontina/",
    "Torneira Eletrica": "https://fogoesshop.com.br/fshop-cozinha/torneira-eletrica/",
    "Metais p/ Cozinha": "https://fogoesshop.com.br/fshop-cozinha/fshop-metais-p-cozinhas/",
    "Promocoes Cozinha, sale, Black Friday": "https://fogoesshop.com.br/fshop-cozinha/promocoes-cozinha/",
    "Torneira filtro": "https://fogoesshop.com.br/fshop-cozinha/fshop-torneira-filtro-e-filtro/",
    "Iluminacao": "https://fogoesshop.com.br/fshop-eletrica/iluminacao-lampada-luminaria/",
    "Tomadas e Interruptores": "https://fogoesshop.com.br/fshop-eletrica/tomada-interruptor-placa/",
    "Fios e Cabos": "https://fogoesshop.com.br/fshop-eletrica/cabos-flexivel-sil/",
    "Conduletes e Conexoes": "https://fogoesshop.com.br/fshop-eletrica/conduletes/",
    "Disjuntores e Armacoes": "https://fogoesshop.com.br/fshop-eletrica/disjuntores/",
    "Promocoes Tintas, sales tinta": "https://fogoesshop.com.br/fshop-tintas/promocoes-tintas/",
    "Pre-Pintura": "https://fogoesshop.com.br/fshop-tintas/pre-pintura/",
    "Ferramentas E Acessorios": "https://fogoesshop.com.br/fshop-tintas/ferramentas-acessorios-trinchas-rolos/",
    "Adesivos, Colas, Selantes e Silicones": "https://fogoesshop.com.br/fshop-tintas/Adesivos-colas-selantes-silicones/",
    "Solventes e Removedores": "https://fogoesshop.com.br/fshop-tintas/solventes-removedores/",
    "Acessibilidade": "https://fogoesshop.com.br/fshop-outros/fshop-ACESSIBILIDADE/",
    "Complementares": "https://fogoesshop.com.br/fshop-outros/fshop-complementares/",
    "Jardim": "https://fogoesshop.com.br/fshop-outros/jardim-amanco/",
    "Tanques e Area Externa": "https://fogoesshop.com.br/fshop-outros/fshop-tanques-area-externa/",
    "Outlet": "https://fogoesshop.com.br/fshop-outros/outlet/"
}

# Cabeçalhos para simular um navegador
headers = {'User-Agent': 'Mozilla/5.0'}

# Chamar a função para analisar os links
analyze_xml_links(links, headers)

# Definir o caminho do diretório base de destino no Google Drive
base_destination_dir = '/content/drive2/MyDrive/WebScraping/FogoesShop/WebScraping_coletaFogoesShop'

# Criar o caminho do novo diretório com a data de hoje
today = date.today().strftime('%Y-%m-%d')
destination_dir = os.path.join(base_destination_dir, today)

# Verificar se o diretório já existe
if not os.path.exists(destination_dir):
    os.makedirs(destination_dir, exist_ok=True)
    # Caminho do diretório de origem
    source_dir = '/content/output'

    # Mover todos os arquivos do diretório de origem para o diretório de destino
    for file_name in os.listdir(source_dir):
        full_file_name = os.path.join(source_dir, file_name)
        if os.path.isfile(full_file_name):
            shutil.move(full_file_name, destination_dir)

    # Mensagem de sucesso em verde
    print(f"\033[92mARQUIVO CRIADO {today}\033[0m")
    print(f"\033[92mARQUIVOS COLETADOS FORAM ENVIADOS PARA O GOOGLE DRIVE COM SUCESSO!\033[0m")
else:
    print(f"\033[91mDiretório para {today} já existe. Nenhum arquivo será movido.\033[0m")

Drive already mounted at /content/drive2; to attempt to forcibly remount, call drive.mount("/content/drive2", force_remount=True).
Categoria: Acessorios Banheiro
Link: https://fogoesshop.com.br/acessorios-banheiros-portatoalha-papeleira-prateleira-cabide/acessorios-banheiro/
Procurando produtos na página 1...


<ipython-input-3-a15acba17df1>:67: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  next_page_link = pagination.find("li", text=">")


Procurando produtos na página 2...
Procurando produtos na página 3...
Procurando produtos na página 4...
Procurando produtos na página 5...
Procurando produtos na página 6...
Procurando produtos na página 7...
Procurando produtos na página 8...
Procurando produtos na página 9...
Procurando produtos na página 10...
Procurando produtos na página 11...
Procurando produtos na página 12...
Procurando produtos na página 13...
Procurando produtos na página 14...
Procurando produtos na página 15...
Procurando produtos na página 16...
Procurando produtos na página 17...
Última página alcançada: 17.
Dados do produto salvos em Acessorios_Banheiro.csv

Categoria: Modelos de Deca
Link: https://fogoesshop.com.br/acessorios-banheiros-portatoalha-papeleira-prateleira-cabide/acessorios-modelos-deca/
Procurando produtos na página 1...
Procurando produtos na página 2...
Procurando produtos na página 3...
Procurando produtos na página 4...
Procurando produtos na página 5...
Procurando produtos na página 6